In [ ]:
import os
import json
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from torch.nn.functional import one_hot
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from torch.optim import AdamW
from torch.optim import lr_scheduler
from torch.amp import autocast
from torch.nn.utils.rnn import pad_sequence
from torchvision import transforms
from torchvision.transforms import v2
import torchvision

from src.Swin_classif import SwinTransformerClassification

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. Install the Kaggle library
!pip install -q kaggle

# 3. Download the dataset directly
!kaggle datasets download -d ambityga/imagenet100

# 4. Unzip the downloaded file
!unzip -q imagenet100.zip -d imagenet100
!rm imagenet100.zip

Dataset URL: https://www.kaggle.com/datasets/ambityga/imagenet100
License(s): unknown
100% 16.1G/16.1G [02:28<00:00, 116MB/s]



In [ ]:
labels = json.load(open("/content/imagenet100/Labels.json"))

# Load training data
images = []
for subset_dirname in os.listdir('/content/imagenet100'):
  if subset_dirname != "Labels.json" and subset_dirname != "val.X":
    for dirname in os.listdir(f'/content/imagenet100/{subset_dirname}'):
      for filename in os.listdir(f'/content/imagenet100/{subset_dirname}/{dirname}'):
        images.append({
            'path': f'/content/imagenet100/{subset_dirname}/{dirname}/{filename}',
            'label': labels[dirname],
        })

df_training = pd.DataFrame(images)
print(f"Total training images: {len(df_training)}")
print(f"top 3 training samples: {df_training[:3]}")

# Load validation data
images = []
for label in os.listdir(f'/content/imagenet100/val.X'):
    for filename in os.listdir(f'/content/imagenet100/val.X/{label}'):
        images.append({
            'path': f'/content/imagenet100/val.X/{label}/{filename}',
            'label': labels[label],
        })

df_val = pd.DataFrame(images)
print(f"Total val images: {len(df_val)}")
print(f"top 3 val samples: {df_val[:3]}")

Total training images: 130000
top 3 training samples:                                                 path  \
0  /content/imagenet100/train.X4/n01819313/n01819...   
1  /content/imagenet100/train.X4/n01819313/n01819...   
2  /content/imagenet100/train.X4/n01819313/n01819...   

                                               label  
0  sulphur-crested cockatoo, Kakatoe galerita, Ca...  
1  sulphur-crested cockatoo, Kakatoe galerita, Ca...  
2  sulphur-crested cockatoo, Kakatoe galerita, Ca...  
Total val images: 5000
top 3 val samples:                                                 path  \
0  /content/imagenet100/val.X/n01614925/ILSVRC201...   
1  /content/imagenet100/val.X/n01614925/ILSVRC201...   
2  /content/imagenet100/val.X/n01614925/ILSVRC201...   

                                               label  
0  bald eagle, American eagle, Haliaeetus leucoce...  
1  bald eagle, American eagle, Haliaeetus leucoce...  
2  bald eagle, American eagle, Haliaeetus leucoce...  


In [ ]:
class Imagenet100Dataset(Dataset):
  def __init__(self, imagenet100_dataset, mapping_label_id, split, image_size=(224, 224)):
    super(Imagenet100Dataset, self).__init__()
    self.imagenet100_dataset = imagenet100_dataset
    self.mapping_label_id = mapping_label_id
    self.n_class = len(mapping_label_id)
    self.image_size = image_size

    # self.transform = transforms.Compose([
    #     transforms.Resize((512, 512)),
    #     transforms.ToTensor()
    # ])
    if split == 'train':
        self.transform = v2.Compose([
            v2.RandomResizedCrop(self.image_size, scale=(0.6, 1.0)),
            v2.RandomHorizontalFlip(p=0.5),
            v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    else:
        self.transform = v2.Compose([
            v2.Resize(self.image_size),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

  def __len__(self):
    return len(self.imagenet100_dataset)

  def __getitem__(self, idx):
    image_path = self.imagenet100_dataset.iloc[idx]['path']
    image = Image.open(image_path).convert("RGB")

    label = self.imagenet100_dataset.iloc[idx]['label']
    id = torch.tensor([self.mapping_label_id[label]], dtype=torch.long)

    return self.transform(image), id

In [ ]:
def custom_collate_fn(batch):
    batch = [item for item in batch if item[0] is not None]
    if not batch:
        return None, None

    images, ids = zip(*batch)

    return torch.stack(images), torch.stack(ids)

In [ ]:
# training data
batch_size = 128
mapping_label_id = {}
mapping_id_label = {}
for i, label in enumerate(labels.values()):
  mapping_label_id[label] = i
  mapping_id_label[i] = label

training_dataset = Imagenet100Dataset(df_training, mapping_label_id=mapping_label_id, split="train", image_size=(224,224))
validating_dataset = Imagenet100Dataset(df_val, mapping_label_id=mapping_label_id, split="val", image_size=(224,224))

training_dataloader = DataLoader(training_dataset, batch_size = batch_size,  collate_fn = custom_collate_fn, shuffle=True, drop_last=True, num_workers=4)
validating_dataloader = DataLoader(validating_dataset, batch_size = batch_size,  collate_fn = custom_collate_fn, shuffle=False, drop_last=False, num_workers=4)

# Model

In [ ]:
model = SwinTransformerClassification(
    backbone_weights_path = None,
    backbone_input_size = (224, 224),
    backbone_patch_size = 4,
    backbone_patch_merging_ratio = 2,
    backbone_in_channels = 3,
    backbone_layers = [2, 2, 6, 2],
    backbone_query_size = 32,
    backbone_n_heads = [2, 4, 8, 16],
    backbone_mlp_factor=4,
    backbone_window_size = 7,
    n_classes = 100,
).to(device)

# TRAINING

In [ ]:
def train_one_epoch(i,  model, device, loader, optimizer, scheduler, max_learning_rate, output_dir, accumulation_step=8):
    model.train()
    losses_displayed = []
    total_losses = []
    step = 0
    optimizer.zero_grad()

    for image_src, ids in loader:
        step += 1

        image_src = image_src.to(device)
        ids = ids.to(device)

        with autocast(device_type=device, dtype=torch.bfloat16):
            pred = model.forward(image_src)
        loss = torch.nn.functional.cross_entropy(pred, ids.squeeze())

        loss = loss / accumulation_step
        loss.backward()
        if step%accumulation_step==0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        losses_displayed.append(loss.item())
        total_losses.append(loss.item())

        if step%100 == 0:
            print(f"training loss mean on last 100 steps : {sum(losses_displayed)/len(losses_displayed)}")
            losses_displayed = []

    print(f"total training loss on the epoch : {sum(total_losses)/len(total_losses)}")
    FINE_TUNED_MODEL = output_dir + "/Swin_classif_epoch_"+ str(i)+".pt"
    torch.save(model.state_dict(), FINE_TUNED_MODEL)

    return model, optimizer, scheduler

In [ ]:
def evaluate(model, device, loader):
    model.eval()
    losses_displayed = []
    total_losses = []

    for image_src, ids in loader:
        image_src = image_src.to(device)
        ids = ids.to(device)

        with torch.no_grad():
          with autocast(device_type=device, dtype=torch.bfloat16):
              pred = model.forward(image_src)
          loss = torch.nn.functional.cross_entropy(pred, ids.squeeze())

        losses_displayed.append(loss.item())
        total_losses.append(loss.item())

    print(f"total validation loss on the epoch : {str(sum(total_losses)/len(total_losses))}")
    return sum(total_losses) / len(total_losses)

## Warmup

In [ ]:
n_epochs = 20
max_learning_rate = 0.0001

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=5e-4,
    steps_per_epoch=len(training_dataloader),
    epochs=n_epochs,
    pct_start=0.2,
    anneal_strategy='cos'
)

output_dir = "/content/drive/MyDrive/output_model"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
losses_val = []

model.train()
for i in range(n_epochs):
    print(f"Epoch {i+1}")
    model, optimizer, scheduler = train_one_epoch(i, model, str(device), training_dataloader, optimizer, scheduler, max_learning_rate, output_dir, accumulation_step=1)
    print(f"Evaluation ")
    loss = evaluate(model, str(device), validating_dataloader)
    losses_val.append(loss)


Epoch 1
training loss mean on last 100 steps : 4.75375
training loss mean on last 100 steps : 4.6134375
training loss mean on last 100 steps : 4.5428125
training loss mean on last 100 steps : 4.508125
training loss mean on last 100 steps : 4.4878125
training loss mean on last 100 steps : 4.450625
training loss mean on last 100 steps : 4.425
training loss mean on last 100 steps : 4.4115625
training loss mean on last 100 steps : 4.4090625
training loss mean on last 100 steps : 4.396875
total training loss on the epoch : 4.498614532019705
Evaluation 
total validation loss on the epoch : 4.28984375
Epoch 2
training loss mean on last 100 steps : 4.37625
training loss mean on last 100 steps : 4.39
training loss mean on last 100 steps : 4.390625
training loss mean on last 100 steps : 4.34375
training loss mean on last 100 steps : 4.288125
training loss mean on last 100 steps : 4.22390625
training loss mean on last 100 steps : 4.1959375
training loss mean on last 100 steps : 4.11296875
trainin

In [ ]:
model.load_state_dict(torch.load("/content/drive/MyDrive/output_model/Swin_classif_epoch_19.pt"))

<All keys matched successfully>

In [ ]:
# training data
batch_size = 32
mapping_label_id = {}
mapping_id_label = {}
for i, label in enumerate(labels.values()):
  mapping_label_id[label] = i
  mapping_id_label[i] = label

training_dataset = Imagenet100Dataset(df_training, mapping_label_id=mapping_label_id, split="train", image_size=(512,512))
validating_dataset = Imagenet100Dataset(df_val, mapping_label_id=mapping_label_id, split="val", image_size=(512,512))

training_dataloader = DataLoader(training_dataset, batch_size = batch_size,  collate_fn = custom_collate_fn, shuffle=True, drop_last=True, num_workers=4)
validating_dataloader = DataLoader(validating_dataset, batch_size = batch_size,  collate_fn = custom_collate_fn, shuffle=False, drop_last=False, num_workers=4)

In [ ]:
n_epochs = 40
max_learning_rate = 0.0001

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=5e-4,
    steps_per_epoch=len(training_dataloader),
    epochs=n_epochs,
    pct_start=0.0,
    anneal_strategy='cos'
)

output_dir = "/content/drive/MyDrive/output_model"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
losses_val = []
for i in range(20, 20 + n_epochs):
    print(f"Epoch {i+1}")
    model, optimizer, scheduler = train_one_epoch(i, model, str(device), training_dataloader, optimizer, scheduler, max_learning_rate, output_dir, accumulation_step=1)
    print(f"Evaluation ")
    loss = evaluate(model, str(device), validating_dataloader)
    losses_val.append(loss)


Epoch 21
training loss mean on last 100 steps : 2.01984375
training loss mean on last 100 steps : 2.012109375
training loss mean on last 100 steps : 1.973984375
training loss mean on last 100 steps : 2.014375
training loss mean on last 100 steps : 2.026796875
training loss mean on last 100 steps : 2.07609375
training loss mean on last 100 steps : 2.02703125
training loss mean on last 100 steps : 2.01359375
training loss mean on last 100 steps : 2.022734375
training loss mean on last 100 steps : 2.073359375
training loss mean on last 100 steps : 2.00015625
training loss mean on last 100 steps : 2.0265625
training loss mean on last 100 steps : 2.069921875
training loss mean on last 100 steps : 2.03546875
training loss mean on last 100 steps : 1.980078125
training loss mean on last 100 steps : 2.009609375
training loss mean on last 100 steps : 1.976015625
training loss mean on last 100 steps : 1.984375
training loss mean on last 100 steps : 2.102578125
training loss mean on last 100 steps

In [ ]:
model.load_state_dict(torch.load("/content/drive/MyDrive/output_model/Swin_classif_epoch_24.pt"))

<All keys matched successfully>

In [ ]:
n_epochs = 40
max_learning_rate = 0.0001

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=5e-4,
    steps_per_epoch=len(training_dataloader),
    epochs=n_epochs,
    pct_start=0.0,
    anneal_strategy='cos'
)

output_dir = "/content/drive/MyDrive/output_model"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
losses_val = []
for i in range(25, 25 + n_epochs):
    print(f"Epoch {i+1}")
    model, optimizer, scheduler = train_one_epoch(i, model, str(device), training_dataloader, optimizer, scheduler, max_learning_rate, output_dir, accumulation_step=1)
    print(f"Evaluation ")
    loss = evaluate(model, str(device), validating_dataloader)
    losses_val.append(loss)


Epoch 26
training loss mean on last 100 steps : 1.37546875
training loss mean on last 100 steps : 1.4451953125
training loss mean on last 100 steps : 1.451953125
training loss mean on last 100 steps : 1.4519140625
training loss mean on last 100 steps : 1.4446875
training loss mean on last 100 steps : 1.440859375
training loss mean on last 100 steps : 1.4553125
training loss mean on last 100 steps : 1.474765625
training loss mean on last 100 steps : 1.4771484375
training loss mean on last 100 steps : 1.4014453125
training loss mean on last 100 steps : 1.4646875
training loss mean on last 100 steps : 1.5001953125
training loss mean on last 100 steps : 1.4218359375
training loss mean on last 100 steps : 1.49546875
training loss mean on last 100 steps : 1.423046875
training loss mean on last 100 steps : 1.467421875
training loss mean on last 100 steps : 1.5004296875
training loss mean on last 100 steps : 1.4323046875
training loss mean on last 100 steps : 1.500859375
training loss mean on 

KeyboardInterrupt: 

In [ ]:
def accuracy(model, device, loader):
    model.eval()
    preds = []
    gt_ids = []

    for image_src, ids in loader:

        image_src = image_src.to(device)
        ids = ids.to(device)

        with torch.no_grad():
          with autocast(device_type=device, dtype=torch.bfloat16):
              pred = model.forward(image_src)
          pred = torch.argmax(pred, dim=1)
          preds.append(pred)
          gt_ids.append(ids.squeeze())

    preds = torch.cat(preds)
    gt_ids = torch.cat(gt_ids)
    correct = (preds == gt_ids).sum().item()
    accuracy = correct / len(gt_ids)
    return accuracy

In [ ]:
image_src, ids = next(iter(validating_dataloader))
image_src = image_src.to(device)
ids = ids.to(device)

with torch.no_grad():
  with autocast(device_type="cuda", dtype=torch.bfloat16):
      pred = model.forward(image_src)

pred.shape, torch.argmax(pred, dim=1).shape, ids.squeeze().shape

(torch.Size([32, 100]), torch.Size([32]), torch.Size([32]))

In [ ]:
accuracy(model, str(device), validating_dataloader)

0.7506

In [ ]:
torch.save(model.backbone.state_dict(), "/content/drive/MyDrive/output_model/Swin_backbone_pretrained.pt")